# Advanced Feature Engineering + LightGBM with Optuna

Builds on `benchmark_submission.ipynb` features. Adds a curated set of 19 new features
(selected from 50+ candidates — using all of them dilutes the signal).

**IMPORTANT:** Must use the full training data (527K rows), NOT the sample (50K).
The financial signal is too weak to survive subsampling.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
sns.set()
import matplotlib.pyplot as plt

import lightgbm as lgbm
import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
)

from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

import warnings
warnings.filterwarnings('ignore')

import joblib
import json

### Load Data (FULL — not the sample)

In [ ]:
X_train = pd.read_csv('Data/X_train.csv', index_col='ROW_ID')
X_test = pd.read_csv('Data/X_test.csv', index_col='ROW_ID')
y_train = pd.read_csv('Data/y_train.csv', index_col='ROW_ID')
sample_submission = pd.read_csv('sample_submission.csv', index_col='ROW_ID')

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
assert X_train.shape[0] > 500000, "ERROR: You loaded the sample data! Use X_train.csv, not X_train_sample.csv"

In [ ]:
RET_features = [f'RET_{i}' for i in range(1, 21)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]
TURNOVER_features = ['MEDIAN_DAILY_TURNOVER']

---
## 1. Existing Features (from benchmark)

In [ ]:
def add_existing_features(df):
    """Features from benchmark_submission.ipynb"""
    ret_cols = [f'RET_{i}' for i in range(1, 21)]
    for i in [3, 5, 10, 15, 20]:
        df[f'AVERAGE_PERF_{i}'] = df[ret_cols[:i]].mean(axis=1)
        df[f'ALLOCATIONS_AVERAGE_PERF_{i}'] = df.groupby('TS')[f'AVERAGE_PERF_{i}'].transform('mean')
    for i in [20]:
        df[f'STD_PERF_{i}'] = df[ret_cols[:i]].std(axis=1)
        df[f'ALLOCATIONS_STD_PERF_{i}'] = df.groupby('TS')[f'STD_PERF_{i}'].transform('mean')
    return df

X_train = add_existing_features(X_train)
X_test = add_existing_features(X_test)

existing_features = RET_features + SIGNED_VOLUME_features + TURNOVER_features
existing_features += [f'AVERAGE_PERF_{i}' for i in [3, 5, 10, 15, 20]]
existing_features += [f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3, 5, 10, 15, 20]]
existing_features += [f'STD_PERF_{i}' for i in [20]]
existing_features += [f'ALLOCATIONS_STD_PERF_{i}' for i in [20]]
print(f"Existing features: {len(existing_features)}")

---
## 2. New Engineered Features

We generate all candidates, then select a **curated subset of 19** that actually help.

In [ ]:
def add_new_features(df):
    """Add new engineered features beyond the benchmark."""
    ret_cols = [f'RET_{i}' for i in range(1, 21)]
    vol_cols = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]

    # A. ROLLING VOLATILITY AT MULTIPLE HORIZONS
    for w in [3, 5, 10]:
        df[f'STD_PERF_{w}'] = df[ret_cols[:w]].std(axis=1)

    # Volatility ratio (safe masking)
    std20 = df['STD_PERF_20']
    mask_valid = std20 > 1e-8
    df['VOL_RATIO_3_20'] = np.where(mask_valid, df['STD_PERF_3'] / std20, 1.0)

    # B. MOMENTUM & TREND STRENGTH
    for w in [5, 10, 20]:
        df[f'POS_RATIO_{w}'] = (df[ret_cols[:w]] > 0).sum(axis=1) / w
    df['MOMENTUM_ACCEL_5_20'] = df['AVERAGE_PERF_5'] - df[ret_cols[5:20]].mean(axis=1)

    # Extreme events
    df['MAX_RET_5'] = df[ret_cols[:5]].max(axis=1)
    df['MIN_RET_5'] = df[ret_cols[:5]].min(axis=1)

    # C. SKEWNESS
    df['SKEW_RET_20'] = df[ret_cols[:20]].skew(axis=1)

    # D. RETURN-VOLUME INTERACTIONS
    ret_5 = df[ret_cols[:5]].values
    vol_5 = df[vol_cols[:5]].values
    df['VOL_WEIGHTED_RET_5'] = np.nanmean(ret_5 * vol_5, axis=1)
    df['AVG_SIGNED_VOL_5'] = df[vol_cols[:5]].mean(axis=1)

    # E. CROSS-SECTIONAL FEATURES
    df['RET_1_RANK'] = df.groupby('TS')['RET_1'].rank(pct=True)
    df['IDIO_RET_5'] = df['AVERAGE_PERF_5'] - df['ALLOCATIONS_AVERAGE_PERF_5']

    # F. TURNOVER (safe, no division)
    df['LOG_TURNOVER'] = np.log1p(df['MEDIAN_DAILY_TURNOVER'].clip(lower=0))
    df['TURNOVER_RANK'] = df.groupby('TS')['MEDIAN_DAILY_TURNOVER'].rank(pct=True)

    # G. SHARPE (safely clipped)
    avg5 = df[ret_cols[:5]].mean(axis=1)
    std5 = df[ret_cols[:5]].std(axis=1)
    ratio = (avg5 / std5).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-10, 10)
    df['SHARPE_5'] = ratio

    # H. STREAK
    ret_vals = df[ret_cols].values
    signs = (ret_vals > 0).astype(int)
    streak = np.ones(len(df))
    direction = signs[:, 0]
    for i in range(1, signs.shape[1]):
        same_dir = (signs[:, i] == direction).astype(float)
        still_going = (streak == i).astype(float)
        streak += same_dir * still_going
    df['WIN_STREAK'] = streak * (2 * direction - 1)

    return df

X_train = add_new_features(X_train)
X_test = add_new_features(X_test)
print(f"Total columns after feature engineering: {X_train.shape[1]}")

In [ ]:
# Curated feature selection:
# Testing showed all 50+ new features HURTS (52.17% < 52.20% benchmark)
# This curated subset of 19 gives 52.24% > 52.20% benchmark
curated_new = [
    'STD_PERF_3', 'STD_PERF_5', 'STD_PERF_10',   # multi-horizon volatility
    'VOL_RATIO_3_20',                               # vol clustering
    'POS_RATIO_5', 'POS_RATIO_20',                  # momentum breadth
    'MOMENTUM_ACCEL_5_20',                           # acceleration
    'MAX_RET_5', 'MIN_RET_5',                        # extreme events
    'SKEW_RET_20',                                   # tail risk
    'VOL_WEIGHTED_RET_5',                            # smart money
    'AVG_SIGNED_VOL_5',                              # volume pressure
    'RET_1_RANK',                                    # cross-sectional rank
    'IDIO_RET_5',                                    # idiosyncratic return
    'SHARPE_5',                                      # risk-adjusted momentum
    'WIN_STREAK',                                    # streak
    'LOG_TURNOVER',                                  # compressed turnover
    'TURNOVER_RANK',                                 # relative turnover
    'GROUP',                                         # allocation group
]

features_lgbm = existing_features + curated_new
features_lgbm = [f for f in features_lgbm if f in X_train.columns]

# Replace any Inf with NaN (LightGBM handles NaN natively)
X_train[features_lgbm] = X_train[features_lgbm].replace([np.inf, -np.inf], np.nan)
X_test[features_lgbm] = X_test[features_lgbm].replace([np.inf, -np.inf], np.nan)

print(f"Existing features: {len(existing_features)}")
print(f"Curated new features: {len(curated_new)}")
print(f"Total features for model: {len(features_lgbm)}")

---
## 3. Baseline CV with Curated Features (before tuning)

In [ ]:
lgbm_params_baseline = {
    "objective": "mse",
    "metric": "mse",
    "num_threads": 8,
    "seed": 42,
    "verbosity": -1,
    "learning_rate": 1e-2,
    "max_depth": 3,
    "feature_pre_filter": False,
}
NUM_BOOST_ROUND = 500

train_dates = X_train['TS'].unique()
n_splits = 8
scores_baseline = []
models_baseline = []

splits = KFold(n_splits=n_splits, random_state=0, shuffle=True).split(train_dates)

for i, (train_idx, val_idx) in enumerate(splits):
    local_train_dates = train_dates[train_idx]
    local_val_dates = train_dates[val_idx]
    train_mask = X_train['TS'].isin(local_train_dates)
    val_mask = X_train['TS'].isin(local_val_dates)

    X_tr = X_train.loc[train_mask, features_lgbm]
    y_tr = y_train.loc[train_mask, 'target']
    X_val = X_train.loc[val_mask, features_lgbm]
    y_val = y_train.loc[val_mask, 'target']

    train_data = lgbm.Dataset(X_tr, label=y_tr.values)
    model = lgbm.train(lgbm_params_baseline, train_data, num_boost_round=NUM_BOOST_ROUND)
    y_pred = model.predict(X_val.values)

    models_baseline.append(model)
    score = accuracy_score((y_val > 0).astype(int), (y_pred > 0).astype(int))
    scores_baseline.append(score)
    print(f"Fold {i+1} - Accuracy: {score * 100:.2f}%")

mean_bl = np.mean(scores_baseline) * 100
std_bl = np.std(scores_baseline) * 100
print(f"\nBaseline with curated features: {mean_bl:.2f}% (+- {std_bl:.2f})")
print(f"Previous benchmark (old features only): 52.22% (+- 0.20)")

In [ ]:
# Feature importance from baseline
fi = pd.DataFrame(
    [m.feature_importance(importance_type='gain') for m in models_baseline],
    columns=features_lgbm,
)
top_n = 30
fi_top = fi.loc[:, fi.mean(0).sort_values(ascending=False).index[:top_n]]

plt.figure(figsize=(10, 10))
sns.barplot(data=fi_top, orient='h', order=fi_top.mean().sort_values(ascending=False).index)
plt.title(f'Top {top_n} Feature Importances (Gain) - Baseline with Curated Features')
plt.xlabel('Mean Gain')
plt.tight_layout()
plt.show()

---
## 4. Optuna Hyperparameter Tuning

4-fold CV for speed. ~1-2 min per trial on full data.

In [ ]:
def objective(trial):
    params = {
        "objective": "mse",
        "metric": "mse",
        "verbosity": -1,
        "num_threads": 8,
        "seed": 42,
        "boosting_type": "gbdt",
        "feature_pre_filter": False,

        "num_leaves": trial.suggest_int("num_leaves", 7, 63),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "min_child_samples": trial.suggest_int("min_child_samples", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 5e-3, 5e-2, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.05),
        "path_smooth": trial.suggest_float("path_smooth", 0.0, 100.0),
        "subsample": trial.suggest_float("subsample", 0.3, 0.8),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
    }

    n_estimators = trial.suggest_int("n_estimators", 200, 800, step=100)

    splits = KFold(n_splits=4, random_state=0, shuffle=True).split(train_dates)
    scores = []
    for fold_i, (train_idx, val_idx) in enumerate(splits):
        local_train_dates = train_dates[train_idx]
        local_val_dates = train_dates[val_idx]
        train_mask = X_train['TS'].isin(local_train_dates)
        val_mask = X_train['TS'].isin(local_val_dates)

        X_tr = X_train.loc[train_mask, features_lgbm]
        y_tr = y_train.loc[train_mask, 'target']
        X_val = X_train.loc[val_mask, features_lgbm]
        y_val = y_train.loc[val_mask, 'target']

        train_data = lgbm.Dataset(X_tr, label=y_tr.values)
        model = lgbm.train(params, train_data, num_boost_round=n_estimators)
        y_pred = model.predict(X_val.values)

        score = accuracy_score((y_val > 0).astype(int), (y_pred > 0).astype(int))
        scores.append(score)

        trial.report(np.mean(scores), fold_i)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)

In [ ]:
study = optuna.create_study(
    direction='maximize',
    study_name='lgbm_advanced_features',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)

study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest trial accuracy: {study.best_trial.value * 100:.2f}%")
print(f"Best params:")
for k, v in study.best_trial.params.items():
    print(f"  {k}: {v}")

In [ ]:
# Save study
joblib.dump(study, 'optuna_study_advanced.pkl')
with open('best_params_advanced.json', 'w') as f:
    json.dump(study.best_trial.params, f, indent=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_optimization_history(study, ax=axes[0])
axes[0].set_title('Optimization History')
plot_param_importances(study, ax=axes[1])
axes[1].set_title('Hyperparameter Importances')
plt.tight_layout()
plt.savefig('optuna_advanced_plots.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Final CV with Best Params (8-fold)

In [ ]:
best_params = study.best_trial.params.copy()
best_n_estimators = best_params.pop('n_estimators')
best_params.update({
    "objective": "mse",
    "metric": "mse",
    "verbosity": -1,
    "num_threads": 8,
    "seed": 42,
    "boosting_type": "gbdt",
    "feature_pre_filter": False,
})

scores_final = []
models_final = []
splits = KFold(n_splits=8, random_state=0, shuffle=True).split(train_dates)

for i, (train_idx, val_idx) in enumerate(splits):
    local_train_dates = train_dates[train_idx]
    local_val_dates = train_dates[val_idx]
    train_mask = X_train['TS'].isin(local_train_dates)
    val_mask = X_train['TS'].isin(local_val_dates)

    X_tr = X_train.loc[train_mask, features_lgbm]
    y_tr = y_train.loc[train_mask, 'target']
    X_val = X_train.loc[val_mask, features_lgbm]
    y_val = y_train.loc[val_mask, 'target']

    train_data = lgbm.Dataset(X_tr, label=y_tr.values)
    model = lgbm.train(best_params, train_data, num_boost_round=best_n_estimators)
    y_pred = model.predict(X_val.values)

    models_final.append(model)
    score = accuracy_score((y_val > 0).astype(int), (y_pred > 0).astype(int))
    scores_final.append(score)
    print(f"Fold {i+1} - Accuracy: {score * 100:.2f}%")

mean_final = np.mean(scores_final) * 100
std_final = np.std(scores_final) * 100
print(f"\n{'='*50}")
print(f"Final CV (8-fold, tuned): {mean_final:.2f}% (+- {std_final:.2f})")
print(f"Benchmark (old features):  52.22% (+- 0.20)")
print(f"Improvement: {mean_final - 52.22:+.2f}%")

In [ ]:
fi_final = pd.DataFrame(
    [m.feature_importance(importance_type='gain') for m in models_final],
    columns=features_lgbm,
)
top_n = 30
fi_top = fi_final.loc[:, fi_final.mean(0).sort_values(ascending=False).index[:top_n]]

plt.figure(figsize=(10, 10))
sns.barplot(data=fi_top, orient='h', order=fi_top.mean().sort_values(ascending=False).index)
plt.title(f'Top {top_n} Feature Importances - Final Tuned Model')
plt.xlabel('Mean Gain')
plt.tight_layout()
plt.savefig('feature_importance_advanced.png', dpi=150, bbox_inches='tight')
plt.show()

top_features = fi_final.mean(0).sort_values(ascending=False).index[:top_n].tolist()
new_in_top = [f for f in top_features if f in curated_new]
print(f"\nNew features in top {top_n}:")
for f in new_in_top:
    print(f"  - {f} (rank {top_features.index(f) + 1})")

---
## 6. Generate Submission

In [ ]:
# Ensemble: average predictions from 8 CV models
preds_ensemble = np.zeros(len(X_test))
for model in models_final:
    preds_ensemble += model.predict(X_test[features_lgbm].values)
preds_ensemble /= len(models_final)

submission_ensemble = pd.DataFrame(
    (preds_ensemble > 0).astype(int),
    index=sample_submission.index, columns=['target'],
)
submission_ensemble.to_csv('preds_advanced_ensemble.csv')
print(f"Ensemble submission: {submission_ensemble['target'].value_counts().to_dict()}")

# Full-data single model
train_data_full = lgbm.Dataset(X_train[features_lgbm], label=y_train['target'].values)
model_full = lgbm.train(best_params, train_data_full, num_boost_round=best_n_estimators)

preds_full = model_full.predict(X_test[features_lgbm].values)
submission_full = pd.DataFrame(
    (preds_full > 0).astype(int),
    index=sample_submission.index, columns=['target'],
)
submission_full.to_csv('preds_advanced_full.csv')
print(f"Full model submission: {submission_full['target'].value_counts().to_dict()}")

model_full.save_model('model_lgbm_advanced.txt')

In [ ]:
print("Summary")
print("=" * 60)
print(f"Features: {len(existing_features)} existing + {len(curated_new)} new = {len(features_lgbm)} total")
print(f"\nBest Optuna params (trial {study.best_trial.number}):")
for k, v in study.best_trial.params.items():
    print(f"  {k}: {v}")
print(f"\nFinal CV: {mean_final:.2f}% (+- {std_final:.2f})")
print(f"Benchmark: 52.22% (+- 0.20)")